## Goal: - 

The goal here is to replicate a small GPT. The idea here is to train on a dataset so that the model starts to produce the next characters based on its learning. This based on all attention is all you need paper.

### Selecting the dataset 
Here the dataset selected the dataset is Tiny Shakespeare Dataset (All of shakespeare concantenated into one file)

In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-01-25 02:19:57--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2026-01-25 02:19:57 (24.8 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()


In [5]:
print("Length of dataset in characters:" , len(text))
print(text[:100])

Length of dataset in charachters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


### Preparing the dataset

We need to tokenize the words (basically convert the text to numbers for AI to understand)

In [6]:
## First lets build the vocabulary -> That means the total number of unique characters present in the dataset

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print("Vocabulary Size:", vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocabulary Size: 65


In [19]:
# Since this is a character level model we are tokenizing all characters

# This is a lookup table
stoi = { ch:i for i,ch in enumerate(chars)} # string to integer
itos = { i:ch for i,ch in enumerate(chars)} # integer to string

encode = lambda s: [stoi[c] for c in s] # encoder: take a string and output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string


## Google uses sentencepiece tokenizer -> A different tokenization technique (sub word unit level)
## OpenAI uses -> Tiktoken (Byte Pair encoding) -> 50000 tokens(approx)


In [20]:
# Because of character level, we have long training sequences
print(encode("hi there"))
print(decode(encode("Hi there")))

[46, 47, 1, 58, 46, 43, 56, 43]
Hi there


### Preparing the dataset

In [11]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch 
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100]) # This is basically the dataset represented in numbers

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [23]:
## Let's split the dataset into train, val and test
#

TRAIN_SPLIT = 0.8
TRAIN_CUT = int(TRAIN_SPLIT * len(data))
print("Train Cut:", TRAIN_CUT)
TRAIN_DATA = data[:TRAIN_CUT]

TEST_SPLIT = 0.1
TEST_CUT = int(TEST_SPLIT * len(data))
print("Test/Val Cut:", TEST_CUT)
VAL_DATA = data[TRAIN_CUT: TRAIN_CUT+ TEST_CUT]
TEST_DATA = data[TRAIN_CUT+ TEST_CUT:]

decode(TRAIN_DATA[:10]), decode(VAL_DATA[:10]), decode(TEST_DATA[:10])

Train Cut: 892315
Test/Val Cut: 111539


KeyError: tensor(18)

(tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47]),
 tensor([], dtype=torch.int64),
 tensor([43, 58,  6,  1, 25, 39, 56, 41, 47, 59]))

### Important note
We don't feed the entire data at once. When we are training such transformers, we randomly sample the chunk of such training set and train on them, of some fixed maximum length(block size/context length)


In [ ]:
block_size = 8
TRAIN_DATA[:block_size+ 1]